# 09 · Recursive queries and hierarchies (Spark)

`employees.manager_id` points to another employee: a **hierarchy** (a tree, like an org chart).
To walk it to any depth you need a **recursive CTE**:

```
WITH RECURSIVE r AS (
    <anchor: the starting rows>
    UNION ALL
    <recursive part: join the table to r, one level deeper>
)
```
It repeats the recursive part until it returns no new rows.

- BigQuery and Spark: `WITH RECURSIVE r AS (...)`
- Oracle: `WITH r (col1, col2, ...) AS (...)`: no word RECURSIVE, but you **must** list the columns.
  Oracle also has its own older syntax: `CONNECT BY PRIOR`.

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../cheatsheets/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`spark/practice/09_recursive_hierarchy.ipynb`](../../spark/practice/09_recursive_hierarchy.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. Numbers 1 to 10  ·  🟢 easy

Make the numbers 1 to 10 (`n`) with a recursive CTE. No table needed.

*Expected result: 10 rows · columns: `n`*

**How it works:** Anchor: the row `1`. Recursive part: `n + 1` from the previous row while `n < 10`. Stops when no new row comes.

In [ ]:
%%sql
WITH RECURSIVE nums AS (
    SELECT 1 AS n
    UNION ALL
    SELECT n + 1 FROM nums WHERE n < 10
)
SELECT n FROM nums ORDER BY n

,n
0,1
1,2
2,3
3,4
4,5
5,6
6,7
7,8
8,9
9,10


**DataFrame API:**

In [ ]:
res = spark.range(1, 11).withColumnRenamed("id", "n")
res.show()

+---+
|  n|
+---+
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
| 10|
+---+

**Why / traps:** The `WHERE n < 10` stops the recursion. Forget it and the query runs until it hits a limit (BigQuery: 500 steps, Spark: 100 by default).

For number or date lists you do not need recursion: `spark.range()` or `SEQUENCE()`.

## Q2. Level of every employee  ·  🟡 medium

Show `emp_id`, `name`, `lvl` for everyone who can be reached from a top boss (an employee with no manager is level 1, their reports level 2, ...).

*Expected result: 30 rows · columns: `emp_id`, `name`, `lvl`*

**How it works:** Anchor = people with no manager (level 1). Each step joins employees whose manager is in the previous level, with `lvl + 1`.

In [ ]:
%%sql
WITH RECURSIVE r AS (
    SELECT emp_id, name, 1 AS lvl
    FROM employees
    WHERE manager_id IS NULL
    UNION ALL
    SELECT e.emp_id, e.name, r.lvl + 1
    FROM employees e
    JOIN r ON e.manager_id = r.emp_id
)
SELECT emp_id, name, lvl FROM r ORDER BY lvl, emp_id

,emp_id,name,lvl
0,1,Alice Johnson,1
1,29,Chloe Collins,1
2,2,Bob Smith,2
3,8,Henry Clark,2
4,14,Nathan King,2
5,18,Rachel Nelson,2
6,22,Victor Roberts,2
7,25,Yara Campbell,2
8,30,Daniel Stewart,2
9,3,Carol White,3


**DataFrame API:**

In [ ]:
# Walk the tree level by level with joins, until no new rows come.
level = employees.filter(F.col("manager_id").isNull()) \
                 .select("emp_id", "name", F.lit(1).alias("lvl"), F.col("name").alias("path"))
all_levels = level
while True:
    nxt = (employees.alias("e")
           .join(level.alias("r"), F.col("e.manager_id") == F.col("r.emp_id"))
           .select("e.emp_id", "e.name", (F.col("r.lvl") + 1).alias("lvl"),
                   F.concat_ws(" > ", "r.path", "e.name").alias("path")))
    if nxt.isEmpty():
        break
    all_levels, level = all_levels.unionByName(nxt), nxt
res = all_levels.select("emp_id", "name", "lvl").orderBy("lvl", "emp_id")
res.show()

+------+--------------+---+
|emp_id|          name|lvl|
+------+--------------+---+
|     1| Alice Johnson|  1|
|    29| Chloe Collins|  1|
|     2|     Bob Smith|  2|
|     8|   Henry Clark|  2|
|    14|   Nathan King|  2|
|    18| Rachel Nelson|  2|
|    22|Victor Roberts|  2|
|    25| Yara Campbell|  2|
|    30|Daniel Stewart|  2|
|     3|   Carol White|  3|
|     4|   David Brown|  3|
|     5|     Eva Green|  3|
|     9|   Irene Lewis|  3|
|    10|   Jack Walker|  3|
|    11|    Karen Hall|  3|
|    15|  Olivia Scott|  3|
|    16|    Paul Adams|  3|
|    19|    Sam Carter|  3|
|    20| Tina Mitchell|  3|
|    23|  Wendy Turner|  3|
+------+--------------+---+
only showing top 20 rows

**Why / traps:** Chloe Collins has no manager, so she is also a level-1 "top". Data has 2 trees.

The DataFrame API has no recursion: loop in Python, one join per level. Fine for shallow trees; each loop adds to the plan, so `cache()` / `checkpoint()` helps when it is deep.

## Q3. Path from the top  ·  🟡 medium

Show `emp_id` and `path` like `Alice Johnson > Bob Smith > Carol White` for everyone in the trees.

*Expected result: 30 rows · columns: `emp_id`, `path`*

**How it works:** Same walk as Q2, but carry text: each step adds `' > ' || name` to the parent's path.

In [ ]:
%%sql
WITH RECURSIVE r AS (
    SELECT emp_id, name, name AS path
    FROM employees
    WHERE manager_id IS NULL
    UNION ALL
    SELECT e.emp_id, e.name, r.path || ' > ' || e.name
    FROM employees e
    JOIN r ON e.manager_id = r.emp_id
)
SELECT emp_id, path FROM r ORDER BY path

,emp_id,path
0,1,Alice Johnson
1,2,Alice Johnson > Bob Smith
2,3,Alice Johnson > Bob Smith > Carol White
3,6,Alice Johnson > Bob Smith > Carol White > Frank Moore
4,7,Alice Johnson > Bob Smith > Carol White > Grace Lee
5,4,Alice Johnson > Bob Smith > David Brown
6,5,Alice Johnson > Bob Smith > Eva Green
7,30,Alice Johnson > Daniel Stewart
8,8,Alice Johnson > Henry Clark
9,9,Alice Johnson > Henry Clark > Irene Lewis


**DataFrame API:**

In [ ]:
# Walk the tree level by level with joins, until no new rows come.
level = employees.filter(F.col("manager_id").isNull()) \
                 .select("emp_id", "name", F.lit(1).alias("lvl"), F.col("name").alias("path"))
all_levels = level
while True:
    nxt = (employees.alias("e")
           .join(level.alias("r"), F.col("e.manager_id") == F.col("r.emp_id"))
           .select("e.emp_id", "e.name", (F.col("r.lvl") + 1).alias("lvl"),
                   F.concat_ws(" > ", "r.path", "e.name").alias("path")))
    if nxt.isEmpty():
        break
    all_levels, level = all_levels.unionByName(nxt), nxt
res = all_levels.select("emp_id", "path").orderBy("path")
res.show()

+------+--------------------+
|emp_id|                path|
+------+--------------------+
|     1|       Alice Johnson|
|     2|Alice Johnson > B...|
|     3|Alice Johnson > B...|
|     6|Alice Johnson > B...|
|     7|Alice Johnson > B...|
|     4|Alice Johnson > B...|
|     5|Alice Johnson > B...|
|    30|Alice Johnson > D...|
|     8|Alice Johnson > H...|
|     9|Alice Johnson > H...|
|    12|Alice Johnson > H...|
|    13|Alice Johnson > H...|
|    10|Alice Johnson > H...|
|    11|Alice Johnson > H...|
|    14|Alice Johnson > N...|
|    15|Alice Johnson > N...|
|    17|Alice Johnson > N...|
|    16|Alice Johnson > N...|
|    18|Alice Johnson > R...|
|    19|Alice Johnson > R...|
+------+--------------------+
only showing top 20 rows

## Q4. Everyone under Bob Smith  ·  🟡 medium

Show `emp_id` and `name` of **all** people below Bob Smith (emp_id 2) at any depth (not Bob himself).

*Expected result: 5 rows · columns: `emp_id`, `name`*

**How it works:** Anchor = Bob's direct reports. Each step adds the reports of people already found.

In [ ]:
%%sql
WITH RECURSIVE r AS (
    SELECT emp_id, name FROM employees WHERE manager_id = 2
    UNION ALL
    SELECT e.emp_id, e.name
    FROM employees e
    JOIN r ON e.manager_id = r.emp_id
)
SELECT emp_id, name FROM r ORDER BY emp_id

,emp_id,name
0,3,Carol White
1,4,David Brown
2,5,Eva Green
3,6,Frank Moore
4,7,Grace Lee


**DataFrame API:**

In [ ]:
level = employees.filter("manager_id = 2").select("emp_id", "name")
found = level
while True:
    level = employees.join(level.select(F.col("emp_id").alias("manager_id")), "manager_id") \
                     .select("emp_id", "name")
    if level.isEmpty():
        break
    found = found.unionByName(level)
res = found.orderBy("emp_id")
res.show()

+------+-----------+
|emp_id|       name|
+------+-----------+
|     3|Carol White|
|     4|David Brown|
|     5|  Eva Green|
|     6|Frank Moore|
|     7|  Grace Lee|
+------+-----------+

**Why / traps:** Start (anchor) at Bob's **direct reports**, then go down.

## Q5. The chain of bosses above Grace Lee  ·  🟡 medium

Walk **up** from Grace Lee (emp_id 7) to the top. Show `step` (1 = her direct manager), `emp_id`, `name` of each boss.

*Expected result: 3 rows · columns: `step`, `emp_id`, `name`*

**How it works:** Anchor = Grace's manager (step 1). Each step joins to the manager of the previous person. Stops at a top boss (manager is NULL).

In [ ]:
%%sql
WITH RECURSIVE up AS (
    SELECT 1 AS step, m.emp_id, m.name, m.manager_id
    FROM employees e
    JOIN employees m ON m.emp_id = e.manager_id
    WHERE e.emp_id = 7
    UNION ALL
    SELECT up.step + 1, m.emp_id, m.name, m.manager_id
    FROM up
    JOIN employees m ON m.emp_id = up.manager_id
)
SELECT step, emp_id, name FROM up ORDER BY step

,step,emp_id,name
0,1,3,Carol White
1,2,2,Bob Smith
2,3,1,Alice Johnson


**DataFrame API:**

In [ ]:
lookup = {r.emp_id: (r.name, r.manager_id) for r in employees.collect()}   # tiny table: OK to collect
rows, boss, step = [], lookup[7][1], 1
while boss is not None:
    rows.append((step, boss, lookup[boss][0]))
    boss, step = lookup[boss][1], step + 1
res = spark.createDataFrame(rows, "step int, emp_id int, name string")
res.show()

+----+------+-------------+
|step|emp_id|         name|
+----+------+-------------+
|   1|     3|  Carol White|
|   2|     2|    Bob Smith|
|   3|     1|Alice Johnson|
+----+------+-------------+

**Why / traps:** Going **up** is the same idea: join on `m.emp_id = up.manager_id` instead.

For a tiny lookup table, `collect()` to Python and walk it there. Never collect big tables.

## Q6. Team size per manager (all levels)  ·  🔴 hard

For every manager, show `manager_id`, `name` and `team_size` = number of people below them at **any** depth. Biggest team first.

<details><summary>Hint</summary>

Recursive CTE of pairs (top_manager, employee): anchor = direct reports; recursive part keeps the same top_manager and goes one level down.

</details>

*Expected result: 12 rows · columns: `manager_id`, `name`, `team_size`*

**How it works:** Build all (boss, employee-below) pairs: start with direct pairs, then extend each pair one level down keeping the same boss. Count per boss.

In [ ]:
%%sql
WITH RECURSIVE pairs AS (
    SELECT manager_id AS boss, emp_id
    FROM employees
    WHERE manager_id IS NOT NULL
    UNION ALL
    SELECT p.boss, e.emp_id
    FROM pairs p
    JOIN employees e ON e.manager_id = p.emp_id
)
SELECT p.boss AS manager_id, m.name, COUNT(*) AS team_size
FROM pairs p
JOIN employees m ON m.emp_id = p.boss
GROUP BY p.boss, m.name
ORDER BY team_size DESC, manager_id

,manager_id,name,team_size
0,1,Alice Johnson,28
1,2,Bob Smith,5
2,8,Henry Clark,5
3,14,Nathan King,3
4,18,Rachel Nelson,3
5,25,Yara Campbell,3
6,3,Carol White,2
7,9,Irene Lewis,2
8,22,Victor Roberts,2
9,15,Olivia Scott,1


**DataFrame API:**

In [ ]:
direct = employees.filter(F.col("manager_id").isNotNull()).select(F.col("manager_id").alias("boss"), "emp_id")
pairs, level = direct, direct
while True:
    level = (level.alias("p").join(employees.alias("e"), F.col("e.manager_id") == F.col("p.emp_id"))
             .select("p.boss", "e.emp_id"))
    if level.isEmpty():
        break
    pairs = pairs.unionByName(level)
res = (pairs.groupBy("boss").agg(F.count("*").alias("team_size"))
       .join(employees.select(F.col("emp_id").alias("boss"), "name"), "boss")
       .select(F.col("boss").alias("manager_id"), "name", "team_size")
       .orderBy(F.desc("team_size"), "manager_id"))
res.show()

+----------+--------------+---------+
|manager_id|          name|team_size|
+----------+--------------+---------+
|         1| Alice Johnson|       28|
|         2|     Bob Smith|        5|
|         8|   Henry Clark|        5|
|        14|   Nathan King|        3|
|        18| Rachel Nelson|        3|
|        25| Yara Campbell|        3|
|         3|   Carol White|        2|
|         9|   Irene Lewis|        2|
|        22|Victor Roberts|        2|
|        15|  Olivia Scott|        1|
|        19|    Sam Carter|        1|
|        26|   Zane Parker|        1|
+----------+--------------+---------+

**Why / traps:** Alice: 28 people (everyone except herself and Chloe). That includes Daniel (no department) but not Chloe (another tree).

## Q7. Cycle guard  ·  🟡 medium

If bad data had a loop (A manages B, B manages A), the recursion would never end. Rewrite **Q2 (level of every employee)** so it also stops at **level 10**, and show the deepest `max_level` found.

*Expected result: 1 row · columns: `max_level`*

**How it works:** Add `WHERE r.lvl < 10` in the recursive part: the walk stops at 10 levels even if the data has a loop.

In [ ]:
%%sql
WITH RECURSIVE r AS (
    SELECT emp_id, 1 AS lvl FROM employees WHERE manager_id IS NULL
    UNION ALL
    SELECT e.emp_id, r.lvl + 1
    FROM employees e
    JOIN r ON e.manager_id = r.emp_id
    WHERE r.lvl < 10
)
SELECT MAX(lvl) AS max_level FROM r

,max_level
0,4


**DataFrame API:**

In [ ]:
# Walk the tree level by level with joins, until no new rows come.
level = employees.filter(F.col("manager_id").isNull()) \
                 .select("emp_id", "name", F.lit(1).alias("lvl"), F.col("name").alias("path"))
all_levels = level
while True:  # add a depth limit in real code
    nxt = (employees.alias("e")
           .join(level.alias("r"), F.col("e.manager_id") == F.col("r.emp_id"))
           .select("e.emp_id", "e.name", (F.col("r.lvl") + 1).alias("lvl"),
                   F.concat_ws(" > ", "r.path", "e.name").alias("path")))
    if nxt.isEmpty():
        break
    all_levels, level = all_levels.unionByName(nxt), nxt
res = all_levels.agg(F.max("lvl").alias("max_level"))
res.show()

+---------+
|max_level|
+---------+
|        4|
+---------+

**Why / traps:** A depth limit is the simple guard in every engine.